In [29]:
import pandas as pd
import numpy as np

In [30]:
df = pd.read_csv("release_generation_yearly_global.csv")

print(f"Shape: {df.shape}")
df.head(10)

Shape: (104273, 24)


,Area,ISO 3 code,Year,Area type,Electricity source,Is aggregated source,Generation (TWh),Generation YoY change (TWh),Generation YoY change (%),Share of generation (%),...,Emissions YoY change (%),Share of emissions (%),Emissions intensity (gCO2e/kWh),Continent,Ember region,EU member,OECD member,G20 member,G7 member,ASEAN member
0,ASEAN,NaN,2000,Region,Solar,False,0.070,NaN,NaN,0.016,...,NaN,0.001,42.857,NaN,NaN,False,False,False,False,False
1,ASEAN,NaN,2000,Region,Wind,False,0.000,NaN,NaN,0.000,...,NaN,0.000,NaN,NaN,NaN,False,False,False,False,False
2,ASEAN,NaN,2000,Region,Hydro,False,53.332,NaN,NaN,12.540,...,NaN,0.506,23.757,NaN,NaN,False,False,False,False,False
3,ASEAN,NaN,2000,Region,Bioenergy,False,10.794,NaN,NaN,2.538,...,NaN,0.934,216.787,NaN,NaN,False,False,False,False,False
4,ASEAN,NaN,2000,Region,Other renewables,False,16.928,NaN,NaN,3.980,...,NaN,0.254,37.630,NaN,NaN,False,False,False,False,False
5,ASEAN,NaN,2000,Region,Nuclear,False,0.000,NaN,NaN,0.000,...,NaN,0.000,NaN,NaN,NaN,False,False,False,False,False
6,ASEAN,NaN,2000,Region,Coal,False,108.301,NaN,NaN,25.466,...,NaN,38.012,879.678,NaN,NaN,False,False,False,False,False
7,ASEAN,NaN,2000,Region,Gas,False,165.814,NaN,NaN,38.989,...,NaN,41.481,626.998,NaN,NaN,False,False,False,False,False
8,ASEAN,NaN,2000,Region,Other fossil,False,70.041,NaN,NaN,16.469,...,NaN,18.813,673.191,NaN,NaN,False,False,False,False,False
9,ASEAN,NaN,2000,Region,Wind and solar,True,0.070,NaN,NaN,0.016,...,NaN,0.001,42.857,NaN,NaN,False,False,False,False,False


In [31]:
df.columns = [
    "area",
    "iso_code",
    "year",
    "area_type",
    "electricity_source",
    "is_aggregated_source",
    "generation_twh",
    "generation_yoy_twh",
    "generation_yoy_percent",
    "generation_share_percent",
    "generation_share_yoy_points",
    "capacity_gw",
    "emissions_mtco2e",
    "emissions_yoy_mtco2e",
    "emissions_yoy_percent",
    "emissions_share_percent",
    "emissions_intensity_gco2e_kwh",
    "continent",
    "ember_region",
    "eu_member",
    "oecd_member",
    "g20_member",
    "g7_member",
    "asean_member"
]

print(df.columns.tolist())

['area', 'iso_code', 'year', 'area_type', 'electricity_source', 'is_aggregated_source', 'generation_twh', 'generation_yoy_twh', 'generation_yoy_percent', 'generation_share_percent', 'generation_share_yoy_points', 'capacity_gw', 'emissions_mtco2e', 'emissions_yoy_mtco2e', 'emissions_yoy_percent', 'emissions_share_percent', 'emissions_intensity_gco2e_kwh', 'continent', 'ember_region', 'eu_member', 'oecd_member', 'g20_member', 'g7_member', 'asean_member']


In [32]:
old_df = df.copy()

In [33]:
text_columns = [
    "area",
    "iso_code",
    "area_type",
    "electricity_source",
    "continent",
    "ember_region"
]

for col in text_columns:
    df[col] = df[col].astype("string").str.strip()

df[text_columns] = df[text_columns].replace(
    ["", "nan", "None", "NA", "N/A"],
    pd.NA
)

In [34]:
boolean_columns = [
    "is_aggregated_source",
    "eu_member",
    "oecd_member",
    "g20_member",
    "g7_member",
    "asean_member"
]

for col in boolean_columns:
    df[col] = df[col].astype("boolean")

In [35]:
print("Complete duplicate rows:", df.duplicated().sum())

Complete duplicate rows: 0


In [36]:
key_columns = [
    "area",
    "year",
    "electricity_source"
]

duplicate_keys = df.duplicated(
    subset=key_columns,
    keep=False
)

print("Duplicate logical records:", duplicate_keys.sum())

print(
    df[duplicate_keys]
    .sort_values(key_columns)
    .head(20)
)

Duplicate logical records: 0
Empty DataFrame
Columns: [area, iso_code, year, area_type, electricity_source, is_aggregated_source, generation_twh, generation_yoy_twh, generation_yoy_percent, generation_share_percent, generation_share_yoy_points, capacity_gw, emissions_mtco2e, emissions_yoy_mtco2e, emissions_yoy_percent, emissions_share_percent, emissions_intensity_gco2e_kwh, continent, ember_region, eu_member, oecd_member, g20_member, g7_member, asean_member]
Index: []

[0 rows x 24 columns]


In [37]:
check_columns = [
    "generation_twh",
    "capacity_gw",
    "emissions_mtco2e"
]

for col in check_columns:
    print(
        col,
        "negative values:",
        (df[col] < 0).sum()
    )

generation_twh negative values: 1408
capacity_gw negative values: 0
emissions_mtco2e negative values: 10


In [38]:
invalid_rows = df[
    (df["generation_twh"] < 0) |
    (df["capacity_gw"] < 0) |
    (df["emissions_mtco2e"] < 0)
]

print(invalid_rows)

            area iso_code  year           area_type electricity_source  \
1852     Albania      ALB  2010  Country or economy        Net imports   
1954     Albania      ALB  2016  Country or economy        Net imports   
1988     Albania      ALB  2018  Country or economy        Net imports   
2039     Albania      ALB  2021  Country or economy        Net imports   
2073     Albania      ALB  2023  Country or economy        Net imports   
...          ...      ...   ...                 ...                ...   
103806    Zambia      ZMB  2022  Country or economy        Net imports   
103823    Zambia      ZMB  2023  Country or economy        Net imports   
103840    Zambia      ZMB  2024  Country or economy        Net imports   
104095  Zimbabwe      ZWE  2014  Country or economy        Net imports   
104112  Zimbabwe      ZWE  2015  Country or economy        Net imports   

        is_aggregated_source  generation_twh  generation_yoy_twh  \
1852                   False           -0.9

In [39]:
percentage_columns = [
    "generation_share_percent",
    "generation_share_yoy_points",
    "emissions_share_percent"
]

for col in percentage_columns:
    print("\n", col)
    print("Minimum:", df[col].min())
    print("Maximum:", df[col].max())


 generation_share_percent
Minimum: -88.54
Maximum: 7633.333

 generation_share_yoy_points
Minimum: -3660.0
Maximum: 1140.0

 emissions_share_percent
Minimum: -73.906
Maximum: 173.923


In [40]:
invalid_generation_share = df[
    (df["generation_share_percent"] < 0) |
    (df["generation_share_percent"] > 100)
]

invalid_emissions_share = df[
    (df["emissions_share_percent"] < 0) |
    (df["emissions_share_percent"] > 100)
]

print("Invalid generation shares:",
      len(invalid_generation_share))

print("Invalid emissions shares:",
      len(invalid_emissions_share))

Invalid generation shares: 3722
Invalid emissions shares: 3573


In [41]:
missing = pd.DataFrame({
    "missing_count": df.isnull().sum(),
    "missing_percent": df.isnull().mean() * 100
})

missing = missing.sort_values(
    "missing_percent",
    ascending=False
)

print(missing)

                               missing_count  missing_percent
emissions_yoy_percent                  48756        46.758029
emissions_intensity_gco2e_kwh          40587        38.923787
generation_yoy_percent                 37036        35.518303
capacity_gw                            28376        27.213181
emissions_yoy_mtco2e                    9828         9.425259
emissions_share_percent                 6339         6.079234
emissions_mtco2e                        6307         6.048546
iso_code                                6266         6.009226
ember_region                            6266         6.009226
continent                               6266         6.009226
generation_share_yoy_points             3779         3.624140
generation_yoy_twh                      3745         3.591534
generation_share_percent                  34         0.032607
year                                       0         0.000000
area                                       0         0.000000
area_typ

In [42]:
keep_null = [
    "iso_code",
    "generation_yoy_twh",
    "generation_yoy_percent",
    "capacity_gw",
    "emissions_mtco2e",
    "emissions_yoy_mtco2e",
    "emissions_yoy_percent",
    "emissions_share_percent",
    "emissions_intensity_gco2e_kwh"
]

In [43]:
countries_df = df[
    df["area_type"] == "Country or economy"
].copy()

regions_df = df[
    df["area_type"] == "Region"
].copy()

print("Country records:", len(countries_df))
print("Region records:", len(regions_df))

Country records: 98007
Region records: 6266


In [44]:
print(
    sorted(
        df["electricity_source"]
        .dropna()
        .unique()
    )
)

['Bioenergy', 'Clean', 'Coal', 'Demand', 'Fossil', 'Gas', 'Hydro', 'Hydro, bioenergy and other renewables', 'Net imports', 'Nuclear', 'Other fossil', 'Other renewables', 'Renewables', 'Solar', 'Total generation', 'Wind', 'Wind and solar']


In [45]:
renewable_sources = [
    "Solar",
    "Wind",
    "Hydro",
    "Bioenergy",
    "Other renewables"
]

fossil_sources = [
    "Coal",
    "Gas",
    "Oil",
    "Other fossil"
]

nuclear_sources = [
    "Nuclear"
]

def classify_energy(source):
    if source in renewable_sources:
        return "Renewable"
    elif source in fossil_sources:
        return "Fossil"
    elif source in nuclear_sources:
        return "Nuclear"
    else:
        return "Other"

df["energy_category"] = df[
    "electricity_source"
].apply(classify_energy)

In [46]:
#List of each type of resource
print(f"Renewable: {renewable_sources}\n")
print(f"Fossil: {fossil_sources}\n")
print(f"Nuclear: {nuclear_sources}")

Renewable: ['Solar', 'Wind', 'Hydro', 'Bioenergy', 'Other renewables']

Fossil: ['Coal', 'Gas', 'Oil', 'Other fossil']

Nuclear: ['Nuclear']


In [47]:
df["is_renewable"] = (
    df["energy_category"] == "Renewable"
)

df["is_fossil"] = (
    df["energy_category"] == "Fossil"
)

df["is_low_carbon"] = df[
    "energy_category"
].isin(["Renewable", "Nuclear"])

In [48]:
renewables = df[
    df["is_renewable"]
]

fossil = df[
    df["is_fossil"]
]

In [49]:
min_year = df["year"].min()

df["year_index"] = df["year"] - min_year

In [50]:
df = df.sort_values(
    [
        "area",
        "electricity_source",
        "year"
    ]
).reset_index(drop=True)

In [51]:
renewable_generation = (
    df[df["is_renewable"]]
    .groupby(
        ["area", "iso_code", "year"],
        as_index=False
    )["generation_twh"]
    .sum()
)

renewable_generation = renewable_generation.rename(
    columns={
        "generation_twh":
        "renewable_generation_twh"
    }
)

print(renewable_generation.head())

          area iso_code  year  renewable_generation_twh
0  Afghanistan      AFG  2000                      0.31
1  Afghanistan      AFG  2001                      0.50
2  Afghanistan      AFG  2002                      0.56
3  Afghanistan      AFG  2003                      0.63
4  Afghanistan      AFG  2004                      0.56


In [ ]:
# Use Ember's own "Total generation" row. Summing every source would also add the
# aggregate rows (Renewables, Fossil, Demand, ...) and inflate the total about 4x.
total_generation = (
    df[df["electricity_source"] == "Total generation"]
    .groupby(
        ["area", "iso_code", "year"],
        as_index=False
    )["generation_twh"]
    .sum()
)

total_generation = total_generation.rename(
    columns={
        "generation_twh":
        "total_generation_twh"
    }
)

In [53]:
renewable_summary = renewable_generation.merge(
    total_generation,
    on=["area", "iso_code", "year"],
    how="left"
)

In [54]:
renewable_summary["renewable_share_percent"] = (
    renewable_summary["renewable_generation_twh"]
    / renewable_summary["total_generation_twh"]
) * 100

print(renewable_summary)

             area iso_code  year  renewable_generation_twh  \
0     Afghanistan      AFG  2000                      0.31   
1     Afghanistan      AFG  2001                      0.50   
2     Afghanistan      AFG  2002                      0.56   
3     Afghanistan      AFG  2003                      0.63   
4     Afghanistan      AFG  2004                      0.56   
...           ...      ...   ...                       ...   
5905     Zimbabwe      ZWE  2020                      3.93   
5906     Zimbabwe      ZWE  2021                      6.06   
5907     Zimbabwe      ZWE  2022                      6.02   
5908     Zimbabwe      ZWE  2023                      5.61   
5909     Zimbabwe      ZWE  2024                      5.86   

      total_generation_twh  renewable_share_percent  
0                     2.72                11.397059  
1                     3.94                12.690355  
2                     4.16                13.461538  
3                     5.10             

In [55]:
fossil_generation = (
    df[df["is_fossil"]]
    .groupby(
        ["area", "iso_code", "year"],
        as_index=False
    )["generation_twh"]
    .sum()
)

fossil_generation = fossil_generation.rename(
    columns={
        "generation_twh":
        "fossil_generation_twh"
    }
)

In [56]:
energy_summary = renewable_summary.merge(
    fossil_generation,
    on=["area", "iso_code", "year"],
    how="left"
)

energy_summary["fossil_share_percent"] = (
    energy_summary["fossil_generation_twh"]
    / energy_summary["total_generation_twh"]
) * 100

ML Part begins here -

In [57]:
ml_df = energy_summary.sort_values(
    ["area", "year"]
).copy()

ml_df["generation_lag_1"] = (
    ml_df.groupby("area")[
        "renewable_generation_twh"
    ].shift(1)
)

ml_df["generation_lag_2"] = (
    ml_df.groupby("area")[
        "renewable_generation_twh"
    ].shift(2)
)

ml_df["generation_lag_3"] = (
    ml_df.groupby("area")[
        "renewable_generation_twh"
    ].shift(3)
)

In [58]:
ml_df["renewable_growth_percent"] = (
    ml_df.groupby("area")[
        "renewable_generation_twh"
    ].pct_change() * 100
)

In [ ]:
ml_df["renewable_3yr_growth_percent"] = (
    (
        ml_df["renewable_generation_twh"]
        / ml_df.groupby("area")[
            "renewable_generation_twh"
        ].shift(3)
    ) - 1
) * 100

# Growth from a zero base is +/-inf; treat it as undefined.
growth_cols = ["renewable_growth_percent", "renewable_3yr_growth_percent"]
ml_df[growth_cols] = ml_df[growth_cols].replace([np.inf, -np.inf], np.nan)


In [60]:
ml_df = ml_df.dropna(
    subset=[
        "generation_lag_1",
        "generation_lag_2",
        "generation_lag_3"
    ]
).reset_index(drop=True)

In [61]:
country_key = ["area", "iso_code"]

countries = (
    df[
        [
            "area",
            "iso_code",
            "continent",
            "area_type",
            "ember_region"
        ]
    ]
    .sort_values(country_key, na_position="last")
    .drop_duplicates(subset=country_key)
    .reset_index(drop=True)
)

countries["country_id"] = countries.index + 1

In [62]:
energy_sources = (
    df[
        [
            "electricity_source",
            "energy_category",
            "is_renewable",
            "is_fossil",
            "is_low_carbon",
            "is_aggregated_source"
        ]
    ]
    .sort_values("electricity_source", na_position="last")
    .drop_duplicates(subset=["electricity_source"])
    .reset_index(drop=True)
)

energy_sources["source_id"] = energy_sources.index + 1

In [63]:
country_key = ["area", "iso_code"]
country_lookup = countries[["country_id", *country_key]]
source_lookup = energy_sources[["source_id", "electricity_source"]]

energy_cleaned = df.merge(
    country_lookup,
    on=country_key,
    how="left",
    validate="many_to_one"
).merge(
    source_lookup,
    on="electricity_source",
    how="left",
    validate="many_to_one"
)

energy_summary = energy_summary.merge(
    country_lookup,
    on=country_key,
    how="left",
    validate="many_to_one"
)

ml_df = ml_df.merge(
    country_lookup,
    on=country_key,
    how="left",
    validate="many_to_one"
)

energy_data = energy_cleaned[
    [
        "country_id",
        "source_id",
        "year",
        "generation_twh",
        "generation_yoy_twh",
        "generation_yoy_percent",
        "generation_share_percent",
        "generation_share_yoy_points",
        "capacity_gw",
        "emissions_mtco2e",
        "emissions_yoy_mtco2e",
        "emissions_yoy_percent",
        "emissions_share_percent",
        "emissions_intensity_gco2e_kwh"
    ]
]

In [64]:
energy_data = energy_data[
    [
        "country_id",
        "source_id",
        "year",
        "generation_twh",
        "generation_yoy_twh",
        "generation_yoy_percent",
        "generation_share_percent",
        "generation_share_yoy_points",
        "capacity_gw",
        "emissions_mtco2e",
        "emissions_yoy_mtco2e",
        "emissions_yoy_percent",
        "emissions_share_percent",
        "emissions_intensity_gco2e_kwh"
    ]
]

In [65]:
assert countries["country_id"].is_unique
assert not countries.duplicated(["area", "iso_code"]).any()
assert energy_sources["source_id"].is_unique
assert energy_sources["electricity_source"].is_unique

country_ids = set(countries["country_id"])
source_ids = set(energy_sources["source_id"])

for table in [energy_cleaned, energy_summary, ml_df, energy_data]:
    assert table["country_id"].notna().all()
    assert set(table["country_id"]).issubset(country_ids)

for table in [energy_cleaned, energy_data]:
    assert table["source_id"].notna().all()
    assert set(table["source_id"]).issubset(source_ids)

assert not energy_data.duplicated(
    ["country_id", "source_id", "year"]
).any()

print("Primary and foreign key checks passed.")

Primary and foreign key checks passed.


In [66]:
import os

os.makedirs("processed", exist_ok=True)
os.makedirs("predictions", exist_ok=True)

In [67]:
energy_cleaned.to_csv(
    "processed/energy_cleaned.csv",
    index=False
)

countries.to_csv(
    "processed/countries.csv",
    index=False
)

energy_sources.to_csv(
    "processed/energy_sources.csv",
    index=False
)

energy_data.to_csv(
    "processed/energy_data.csv",
    index=False
)

energy_summary.to_csv(
    "processed/energy_summary.csv",
    index=False
)

ml_df.to_csv(
    "processed/ml_dataset.csv",
    index=False
)

In [68]:
print("Cleaned dataset:",
      df.shape)

print("Countries:",
      countries.shape)

print("Energy sources:",
      energy_sources.shape)

print("Energy data:",
      energy_data.shape)

print("Energy summary:",
      energy_summary.shape)

print("ML dataset:",
      ml_df.shape)

Cleaned dataset: (104273, 29)
Countries: (224, 6)
Energy sources: (17, 7)
Energy data: (104273, 14)
Energy summary: (5910, 9)
ML dataset: (5283, 14)


In [69]:
print(
    os.listdir("processed")
)

['countries.csv', 'energy_cleaned.csv', 'energy_data.csv', 'energy_sources.csv', 'energy_summary.csv', 'ml_dataset.csv']
